# Optimal transport on surfaces

Optimal transport (OT) asks how to move one distribution into another at minimum cost. In graphics, two important mesh formulations are Earth Mover's Distance (EMD) and Benamou-Brenier.

In this notebook we study both as discrete convex problems:

- **EMD on surfaces** by Solomon, Rustamov, Guibas, Butscher (2014), an SOCP.
- **Dynamical OT on surfaces** Lavenant, Claici, Chien, Solomon (2018), also an SOCP after time discretization.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spl
import matplotlib.pyplot as plt
import cvxpy as cp
import gpytoolbox as gpy

## Earth Mover's Distance on a surface

Given two vertex probability distributions $\rho_0, \rho_1$, EMD solves
$$ \mathrm{EMD}(\rho_0, \rho_1) = \min_J \int_M \|J(x)\|_2\,\mathrm{d}A \quad \text{subject to} \quad \operatorname{div}(J) = \rho_1 - \rho_0. $$
Discretizing with one tangent vector per face gives a sum of face-wise $\ell_2$ norms and linear constraints, so the problem is an SOCP.

We first solve an instance with CVXPY, then compare against a Python implementation of their MATLAB ADMM solver.

### Setup

We load the horse mesh `395.off` from the paper and place two Gaussian distributions near the nose and tail vertices.

In [ ]:
import igl
from src.first_order_fem import first_order_fem

V_horse, F_horse = igl.read_triangle_mesh('../external/emdadmm/395.off')
fem_horse = first_order_fem(V_horse, F_horse)
n_v_horse = V_horse.shape[0]
n_f_horse = F_horse.shape[0]
print(f'horse mesh: {n_v_horse} vertices, {n_f_horse} faces')

nose, tail = 1779, 7943
d_nose = np.linalg.norm(V_horse - V_horse[nose], axis=1)
d_tail = np.linalg.norm(V_horse - V_horse[tail], axis=1)
rho0 = np.exp(-50 * d_nose**2)
rho0 /= rho0.sum()
rho1 = np.exp(-50 * d_tail**2)
rho1 /= rho1.sum()

### Using CVXPY

We now write the SOCP directly in CVXPY. The discrete divergence is the adjoint of the gradient,
$$ \operatorname{div}(J) = -G^\top A_f J, $$
with $G$ the per-component gradient operator and $A_f$ the diagonal matrix of face areas.

In [ ]:
areas = fem_horse['face_areas']
grad = fem_horse['grad']

J = cp.Variable((n_f_horse, 3))
div_J = -sum(grad[i].T @ cp.multiply(areas, J[:, i]) for i in range(3))

objective = cp.Minimize(cp.sum(cp.multiply(areas, cp.norm(J, 2, axis=1))))
constraints = [div_J == rho1 - rho0]
problem = cp.Problem(objective, constraints)
problem.solve()

emd_cvxpy = problem.value
J_cvxpy = J.value
print(f'EMD via CVXPY  : {emd_cvxpy:.6f}')
print(f'CVXPY chose    : {problem.solver_stats.solver_name}')

### Using ADMM

Following Solomon et al., we decompose tangent flow as $J = \nabla a + R\nabla b$, where $R$ is a 90-degree rotation in each tangent plane. Since $\operatorname{div}(R\nabla b)=0$, the constraint reduces to a Poisson equation for $a$.

The remaining variables are solved with ADMM. The ADMM result should be close to the CVXPY value, with the gap controlled by the size of the eigenbasis.

In [ ]:
from src.emd_precompute import emd_precompute
from src.emd_admm import emd_admm

n_eigs = 100
print(f'computing {n_eigs} Laplace-Beltrami eigenvectors...')
evals, evecs = spl.eigsh(-fem_horse['laplacian'], k=n_eigs, M=fem_horse['vtx_inner_prods'], sigma=0.0, which='LM')

print('precomputing EMD structure...')
structure_horse = emd_precompute(V_horse, F_horse, evecs[:, 1:])

print('running ADMM...')
emd_horse, J_horse = emd_admm(rho0, rho1, structure_horse, n_iter=100, verbose=False)

print(f'EMD via CVXPY oracle: {emd_cvxpy:.6f}')
print(f'EMD via ADMM: {emd_horse:.6f}')

The two values are close, with ADMM typically slightly above the CVXPY optimum when using a truncated eigenbasis. Increasing the number of eigenvectors tightens the approximation.

**▶ Your turn.** Try `n_eigs = 50` and compare the ADMM objective against the CVXPY value. Then try `n_eigs = 150` and see if the gap narrows further.

### Visualization

Let's look at the horse mesh with scalar fields $\rho_0$ and $\rho_1$, and the face-based optimal flow field $J$. You can close the window to return to the notebook.

In [ ]:
import polyscope as ps

# ps.init()
# ps.remove_all_structures()
# ps.set_ground_plane_mode('none')
# ps_horse = ps.register_surface_mesh('horse', V_horse, F_horse, smooth_shade=True)
# ps_horse.add_scalar_quantity('rho_0 (nose)', rho0, cmap='viridis', enabled=True)
# ps_horse.add_scalar_quantity('rho_1 (tail)', rho1, cmap='viridis')
# ps_horse.add_vector_quantity('optimal flow J', J_horse, defined_on='faces',
#                               enabled=True, color=(0.9, 0.2, 0.2))
# ps.show()

**▶ Your turn.** Pick a new source-target vertex pair on the horse and rerun `emd_admm` with new Gaussians. You can reuse the same precomputation, so each new query is much cheaper than rebuilding the full system.

## Dynamical optimal transport on surfaces

EMD gives one static transport plan. Dynamical OT instead looks for a full path of densities $\rho(t,\cdot)$ between $\rho_0$ and $\rho_1$ with minimum kinetic action.

Benamou and Brenier (2000) formulate this as
$$ \min_{\rho,m} \int_0^1 \!\!\int_M \frac{\|m\|^2}{2\rho}\,\mathrm{d}A\,\mathrm{d}t \quad \text{s.t.} \quad \partial_t \rho + \operatorname{div}(m)=0, \ \rho(0,\cdot)=\rho_0, \ \rho(1,\cdot)=\rho_1. $$
The kinetic-energy density $\|m\|^2/(2\rho)$ is convex for $\rho>0$, and in CVXPY it is modeled with `quad_over_lin(m, rho)`.

Here we use the hand example and compare CVXPY and ADMM on the same setup.

### Using CVXPY

We load the hand mesh and boundary data, then solve with CVXPY.

In [ ]:
from src.dynamic_ot_cvxpy import dynamic_ot_cvxpy
from src.dynamic_ot_run import dynamic_ot_run
import pathlib
import igl
import time

meshes_dir = pathlib.Path('../external/DynamicalOTSurfaces/ADMM code/meshes').resolve()
mesh_hand = str(meshes_dir / 'hand_3k.off')
bdy_hand = str(meshes_dir / 'hand.bdy')

V_hand, F_hand = igl.read_triangle_mesh(mesh_hand)
mu0_hand = np.loadtxt(str(meshes_dir / 'hand_data_mu0.txt'))
mu1_hand = np.loadtxt(str(meshes_dir / 'hand_data_mu1.txt'))
print(f'hand mesh: {V_hand.shape[0]} vertices, {F_hand.shape[0]} faces')

cvx_start = time.perf_counter()
mu_cvxpy_hand, dual_obj_hand = dynamic_ot_cvxpy(V_hand, F_hand, mu0_hand, mu1_hand, 13, alpha=0.0)
cvx_time = time.perf_counter() - cvx_start
print(f'CVXPY dual objective (n_time=13): {dual_obj_hand:.6f}')
print(f'CVXPY elapsed time: {cvx_time:.3f} s')

## Using ADMM
Now we solve the same example with ADMM and compare objective values and interpolation density.

In [ ]:
admm_start = time.perf_counter()
result_hand = dynamic_ot_run(mesh_path=mesh_hand, boundary_path=bdy_hand, n_time=13, n_iter=500, verbose=False)
admm_time = time.perf_counter() - admm_start
mu_hand = result_hand['mu']
print(f'ADMM primal objective (n_time=13): {result_hand["objective"][-1]:.6f}')
print(f'ADMM elapsed time: {admm_time:.3f} s')
rel_err = np.linalg.norm(mu_cvxpy_hand - mu_hand) / np.linalg.norm(mu_hand)
print(f'CVXPY vs ADMM relative error on mu(t): {rel_err:.3e}')

### Visualization

Let's visualize $\mu(t)$! You can view different times by moving the slider.

In [ ]:
import polyscope as ps
import polyscope.imgui as psim

ps.init()
ps.remove_all_structures()
ps.set_ground_plane_mode('none')
ps_hand = ps.register_surface_mesh('hand', V_hand, F_hand, smooth_shade=True)

n_t = mu_hand.shape[0]
state = {'t': 0}

def show_t(t):
    mu_t = mu_hand[t]
    ps_hand.add_scalar_quantity('mu(t)', mu_t, cmap='blues',\
                                 vminmax=(float(mu_t.min()), float(mu_t.max())), enabled=True)

show_t(0)

def callback():
    changed, t = psim.SliderInt('time step', state['t'], v_min=0, v_max=n_t - 1)
    if changed:
        state['t'] = t
        show_t(t)

ps.set_user_callback(callback)
ps.show()
ps.clear_user_callback()

**▶ Your turn.** Keep `n_T = 13`, rerun the hand solve with a smaller `n_iter` (for example, 20), and inspect the result. You should see a less smooth interpolation and larger primal/dual residuals, illustrating the effect of stopping ADMM early.

## Summary

EMD and dynamical OT both fit naturally into second-order cone formulations. In this notebook we use CVXPY as an oracle for EMD and compare it with an ADMM solver. Similarly, we compare CVXPY and ADMM for an example recovering an interpolant between two distributions on a mesh using the dynamical OT formulation.

## References

[1] *Convolutional Wasserstein Distances: Efficient Optimal Transportation on Geometric Domains*.
       Solomon, J.; Rustamov, R.; Guibas, L.; Butscher, A.
       ACM Transactions on Graphics (SIGGRAPH) 33 (4): 1 (2014).

[2] *Dynamics of optimal transport on discrete surfaces*.
       Lavenant, H.; Claici, S.; Chien, E.; Solomon, J.
       ACM Transactions on Graphics (SIGGRAPH Asia) 37 (6): 1 (2018).

[3] *Numerical computation of the Wasserstein distance*.
       Benamou, J.-D.; Brenier, Y.
       ESAIM: Mathematical Modelling and Numerical Analysis 34 (2): 375 (2000).